# Tracking de la pelotita amarilla — `trayectoria3.MOV`

Este notebook:

1. Lee el video desde  
   `C:\Users\user\Desktop\Labo5\1_Fluidos\datos\crudos\trayectoria3.MOV`
2. Sigue la pelotita amarilla frame a frame.
3. Guarda la trayectoria completa como CSV.
4. Genera un video con un círculo sobre la pelotita y su trayectoria reciente.

Los resultados se guardan automáticamente en:

`C:\Users\user\Desktop\Trayectoria3_resultados`


In [ ]:
# Si te falta alguna librería, corré ESTA celda una sola vez:
# %pip install opencv-python numpy pandas matplotlib


In [ ]:
from pathlib import Path
import math

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

VIDEO = Path(
    r"C:\Users\user\Desktop\Labo5\1_Fluidos\datos\crudos\trayectoria3.MOV"
)

CARPETA_SALIDA = Path(
    r"C:\Users\user\Desktop\Trayectoria3_resultados"
)

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

CSV_SALIDA = CARPETA_SALIDA / "trayectoria3.csv"
VIDEO_SALIDA = CARPETA_SALIDA / "trayectoria3_seguimiento.mp4"

print("Video:", VIDEO)
print("Salida:", CARPETA_SALIDA)


In [ ]:
# ------------------------------------------------------------
# PARÁMETROS DE DETECCIÓN
# ------------------------------------------------------------

# Umbral HSV ajustado para la pelotita amarilla de trayectoria3.MOV
HSV_BAJO = np.array([18, 70, 100], dtype=np.uint8)
HSV_ALTO = np.array([45, 255, 255], dtype=np.uint8)

# Área aproximada admisible de objetos amarillos
AREA_MIN = 300
AREA_MAX = 2400

# Circularidad mínima
CIRCULARIDAD_MIN = 0.25


In [ ]:
def detectar_candidatos(frame):
    """Encuentra regiones amarillas aproximadamente circulares."""

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, HSV_BAJO, HSV_ALTO)

    # Limpieza de ruido
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8),
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        np.ones((7, 7), np.uint8),
    )

    contornos, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    candidatos = []

    for c in contornos:
        area = cv2.contourArea(c)

        if not (AREA_MIN <= area <= AREA_MAX):
            continue

        perimetro = cv2.arcLength(c, True)

        circularidad = (
            4 * math.pi * area / (perimetro**2 + 1e-12)
        )

        if circularidad < CIRCULARIDAD_MIN:
            continue

        M = cv2.moments(c)

        if M["m00"] == 0:
            continue

        x = M["m10"] / M["m00"]
        y = M["m01"] / M["m00"]

        candidatos.append(
            {
                "pos": np.array([x, y], dtype=float),
                "area": float(area),
                "circularidad": float(circularidad),
            }
        )

    return candidatos


In [ ]:
def obtener_trayectoria(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"No pude abrir el video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"FPS = {fps:.3f}")
    print(f"Frames = {n_frames}")
    print(f"Resolución = {ancho} x {alto}")

    centro_imagen = np.array(
        [ancho / 2, alto / 2],
        dtype=float,
    )

    iniciado = False
    pos_anterior = None
    velocidad_pred = np.zeros(2, dtype=float)
    misses = 0

    filas = []

    for i in range(n_frames):

        ok, frame = cap.read()

        if not ok:
            break

        candidatos = detectar_candidatos(frame)
        elegido = None

        # ----------------------------------------------------
        # INICIO DEL TRACKEO
        # ----------------------------------------------------
        if not iniciado:

            candidatos_inicio = []

            for c in candidatos:

                r = np.linalg.norm(
                    c["pos"] - centro_imagen
                )

                # En trayectoria3 la pelotita móvil aparece
                # dentro del recipiente, no sobre el borde.
                if (
                    120 < r < 350
                    and 500 < c["area"] < 1400
                ):
                    candidatos_inicio.append(c)

            if candidatos_inicio:

                elegido = min(
                    candidatos_inicio,
                    key=lambda c: np.linalg.norm(
                        c["pos"] - centro_imagen
                    ),
                )

                pos_anterior = elegido["pos"].copy()
                velocidad_pred[:] = 0
                iniciado = True
                misses = 0

        # ----------------------------------------------------
        # CONTINUACIÓN DEL TRACKEO
        # ----------------------------------------------------
        else:

            # Predicción de posición por velocidad constante
            pred = pos_anterior + velocidad_pred

            puntuados = []

            for c in candidatos:

                distancia = np.linalg.norm(
                    c["pos"] - pred
                )

                # Penaliza candidatos cuyo tamaño no se parece
                # al de la pelotita móvil.
                score = (
                    distancia
                    + 0.004 * abs(c["area"] - 850)
                )

                puntuados.append(
                    (score, distancia, c)
                )

            if puntuados:

                _, distancia, mejor = min(
                    puntuados,
                    key=lambda z: z[0],
                )

                # Ventana máxima de búsqueda.
                gate = (
                    90
                    if misses < 3
                    else min(220, 90 + 20 * misses)
                )

                if distancia < gate:
                    elegido = mejor

            if elegido is not None:

                nueva = elegido["pos"]

                vel_instantanea = (
                    nueva - pos_anterior
                )

                # Suavizado del predictor
                velocidad_pred = (
                    0.65 * velocidad_pred
                    + 0.35 * vel_instantanea
                )

                pos_anterior = nueva.copy()
                misses = 0

            else:

                misses += 1
                pos_anterior = pred.copy()

        # ----------------------------------------------------
        # GUARDAR RESULTADO DEL FRAME
        # ----------------------------------------------------
        if elegido is None:

            filas.append(
                {
                    "frame": i,
                    "t_video_s": i / fps,
                    "x_px": np.nan,
                    "y_px": np.nan,
                    "detectado": False,
                }
            )

        else:

            filas.append(
                {
                    "frame": i,
                    "t_video_s": i / fps,
                    "x_px": float(elegido["pos"][0]),
                    "y_px": float(elegido["pos"][1]),
                    "detectado": True,
                }
            )

    cap.release()

    df = pd.DataFrame(filas)

    validos = df.index[df["detectado"]].tolist()

    if not validos:
        raise RuntimeError(
            "No se detectó la pelotita amarilla."
        )

    # Elimina los frames anteriores a la primera detección
    primero = validos[0]

    df = (
        df.loc[primero:]
        .copy()
        .reset_index(drop=True)
    )

    # Interpolación solamente de huecos internos
    era_nan = (
        df[["x_px", "y_px"]]
        .isna()
        .any(axis=1)
    )

    df["x_px"] = df["x_px"].interpolate(
        limit_area="inside"
    )

    df["y_px"] = df["y_px"].interpolate(
        limit_area="inside"
    )

    df["interpolado"] = (
        era_nan
        & df[["x_px", "y_px"]]
        .notna()
        .all(axis=1)
    )

    frame_inicial = int(df.loc[0, "frame"])

    df["t_desde_deteccion_s"] = (
        df["frame"] - frame_inicial
    ) / fps

    # Coordenada cartesiana: y positivo hacia arriba
    df["y_cart_px"] = alto - df["y_px"]

    return df, fps, ancho, alto


In [ ]:
# ------------------------------------------------------------
# CORRER EL TRACKEO Y GUARDAR CSV
# ------------------------------------------------------------

df, fps, ancho, alto = obtener_trayectoria(VIDEO)

df.to_csv(CSV_SALIDA, index=False)

print()
print("CSV guardado en:")
print(CSV_SALIDA)

print()
print("Cantidad de puntos:", len(df))
print("Frames interpolados:", int(df["interpolado"].sum()))

df.head()


In [ ]:
# ------------------------------------------------------------
# VER LA TRAYECTORIA RÁPIDAMENTE
# ------------------------------------------------------------

plt.figure(figsize=(7, 7))

plt.plot(
    df["x_px"],
    df["y_cart_px"],
    "-",
    linewidth=1,
)

plt.scatter(
    df["x_px"].iloc[0],
    df["y_cart_px"].iloc[0],
    s=60,
    label="Inicio",
)

plt.scatter(
    df["x_px"].iloc[-1],
    df["y_cart_px"].iloc[-1],
    s=60,
    label="Final",
)

plt.xlabel("x [px]")
plt.ylabel("y [px]")
plt.title("Trayectoria de la pelotita amarilla")
plt.axis("equal")
plt.grid()
plt.legend()
plt.show()


In [ ]:
def crear_video_seguimiento(
    video_path,
    df,
    fps,
    ancho,
    alto,
    salida_mp4,
):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"No pude reabrir el video: {video_path}"
        )

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        str(salida_mp4),
        fourcc,
        fps,
        (ancho, alto),
    )

    if not writer.isOpened():
        raise RuntimeError(
            "No se pudo crear el video de salida."
        )

    posiciones = {
        int(row.frame): (
            float(row.x_px),
            float(row.y_px),
        )
        for row in df.itertuples()
        if (
            np.isfinite(row.x_px)
            and np.isfinite(row.y_px)
        )
    }

    historial = []

    # Muestra aproximadamente los últimos 2 segundos
    max_historial = max(
        1,
        int(round(2.0 * fps)),
    )

    i = 0

    while True:

        ok, frame = cap.read()

        if not ok:
            break

        if i in posiciones:

            x, y = posiciones[i]

            p = (
                int(round(x)),
                int(round(y)),
            )

            historial.append(p)

            if len(historial) > max_historial:
                historial.pop(0)

            # Dibuja la trayectoria reciente
            for j in range(1, len(historial)):

                cv2.line(
                    frame,
                    historial[j - 1],
                    historial[j],
                    (0, 255, 0),
                    3,
                )

            # Marca la pelotita
            cv2.circle(
                frame,
                p,
                30,
                (0, 0, 255),
                4,
            )

            cv2.circle(
                frame,
                p,
                4,
                (255, 255, 255),
                -1,
            )

            cv2.putText(
                frame,
                "pelotita",
                (p[0] + 35, p[1] - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )

        writer.write(frame)
        i += 1

    cap.release()
    writer.release()


In [ ]:
# ------------------------------------------------------------
# CREAR VIDEO CON EL SEGUIMIENTO
# ------------------------------------------------------------

crear_video_seguimiento(
    VIDEO,
    df,
    fps,
    ancho,
    alto,
    VIDEO_SALIDA,
)

print("Video guardado en:")
print(VIDEO_SALIDA)
print()
print("Listo.")


## Archivos generados

Al terminar las celdas anteriores vas a tener:

- `C:\Users\user\Desktop\Trayectoria3_resultados\trayectoria3.csv`
- `C:\Users\user\Desktop\Trayectoria3_resultados\trayectoria3_seguimiento.mp4`

En el CSV, `x_px` y `y_px` son las coordenadas de imagen.  
`y_cart_px` es la misma coordenada vertical pero invertida, de forma que \(y\) crezca hacia arriba.
